# ส่วนที่ 4b · วัดผล mT5 ที่เทรนแล้วบน test กลาง (ไม่ต้องเทรนใหม่)

**วิธีใช้ (Colab):** Runtime → Change runtime type → **T4 GPU** แล้ว **Run all** · ใช้เวลาประมาณ 10–15 นาที

ต้องมี 2 ไฟล์ใน Google Drive โฟลเดอร์ `MyDrive/` (หรือแก้ path ในช่องตั้งค่า)
1. `mt5-thaisum.zip` – โมเดลที่เทรนแล้ว
2. `test.jsonl` – test กลางจากคนที่ 1 (`data/processed/test.jsonl`)

ผลลัพธ์: `predictions_mt5.csv` → ส่งให้คนที่ 7 ใส่ในโฟลเดอร์ `predictions/`

In [ ]:
!pip -q install transformers sentencepiece rouge_score pythainlp

In [ ]:
MODEL_ZIP = "/content/drive/MyDrive/mt5-thaisum.zip"
TEST_JSONL = "/content/drive/MyDrive/test.jsonl"
N_TEST = 500         # 500 ข่าวแรก ให้ตรงกับ notebook 07
MAX_INPUT, MAX_TARGET, NUM_BEAMS = 512, 128, 4   # ค่าเดียวกับตอนเทรนใน 04-mt5

In [ ]:
import os, shutil
from google.colab import drive
drive.mount("/content/drive")
if not os.path.exists("mt5-thaisum"):
    shutil.unpack_archive(MODEL_ZIP, "mt5-thaisum")
print(os.listdir("mt5-thaisum"))

In [ ]:
import pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
test_df = pd.read_json(TEST_JSONL, lines=True).head(N_TEST)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("GPU:", torch.cuda.get_device_name(0) if device == "cuda" else "ไม่มี (ช้ามาก ให้เปิด T4 GPU)")
tokenizer = AutoTokenizer.from_pretrained("mt5-thaisum")
model = AutoModelForSeq2SeqLM.from_pretrained("mt5-thaisum").to(device).eval()

def summarize(texts, bs=8):
    out = []
    for i in range(0, len(texts), bs):
        enc = tokenizer(texts[i:i+bs], max_length=MAX_INPUT, truncation=True, padding=True, return_tensors="pt").to(device)
        with torch.no_grad():
            ids = model.generate(**enc, max_new_tokens=MAX_TARGET, num_beams=NUM_BEAMS, no_repeat_ngram_size=3, length_penalty=1.0)
        out += tokenizer.batch_decode(ids, skip_special_tokens=True)
        print(len(out), "/", len(texts))
    return out

preds = summarize(test_df.body.tolist())

In [ ]:
from rouge_score import rouge_scorer
from pythainlp.tokenize import word_tokenize
class ThaiTokenizer:
    def tokenize(self, text):
        return [w for w in word_tokenize(text, engine="newmm") if w.strip()]
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], tokenizer=ThaiTokenizer())
s = [scorer.score(r, p) for p, r in zip(preds, test_df.summary)]
print({k: round(100 * sum(x[k].fmeasure for x in s) / len(s), 2) for k in ["rouge1", "rouge2", "rougeL"]})

out = test_df[["id", "type", "body", "summary"]].assign(pred=preds)
out.to_csv("predictions_mt5.csv", index=False)
shutil.copy("predictions_mt5.csv", "/content/drive/MyDrive/predictions_mt5_shared.csv")
print("บันทึกแล้ว: MyDrive/predictions_mt5_shared.csv")